# bench_figures

> Figures for the Phase 8 benchmark: scaling curves, confirmatory contrasts, recovery, beliefs, secondary metrics, side sweeps and study B.

Spec: `specs/multiagent/phase8/` (design-low-level section 6).

In [ ]:
#| default_exp bench_figures

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib.pyplot as plt
import numpy as np

from explRL.bench_report import CH, _mean, _sel, byN
from explRL.bench_stats import bootCI


def _lines(ax, rows, metric, chans=CH, **kw):
    for ch in chans:
        a = byN(rows, metric, channel=ch, **kw)
        if len(a):
            ax.errorbar(a[:, 0], a[:, 1], yerr=[a[:, 1] - a[:, 2], a[:, 3] - a[:, 1]], marker='o', ms=3, capsize=2, label=ch)
    ax.set_xscale('log', base=2); ax.set_xlabel('team size'); ax.set_title(metric, fontsize=9)


def mainFigures(rows, conf, prereg):
    "Figures for the main grid: scaling, confirmatory, recovery, beliefs, secondary."
    figs, main = {}, _sel(rows, study='main')
    f, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for a, b in zip(ax, ('compute', 'time')):
        _lines(a, main, 'perAgentSecond', budget=b); a.set_title(f'welfare per agent-second, fixed {b}', fontsize=9)
    ax[0].legend(fontsize=7); figs['scaling'] = f
    f, ax = plt.subplots(1, len(prereg['tests']), figsize=(4 * len(prereg['tests']), 3.5))
    for a, (name, t) in zip(np.atleast_1d(ax), prereg['tests'].items()):
        pts = []
        for N in t['sizes']:
            A = {r['seed']: r[t['metric']] for r in _sel(main, channel=t['a'], budget=prereg['budget']) if int(r['N']) == N}
            B = {r['seed']: r[t['metric']] for r in _sel(main, channel=t['b'], budget=prereg['budget']) if int(r['N']) == N}
            d = [A[s] - B[s] for s in A if s in B]
            pts.append((N, _mean(d), *bootCI(d)))
        p = np.array(pts)
        a.errorbar(p[:, 0], p[:, 1], yerr=[p[:, 1] - p[:, 2], p[:, 3] - p[:, 1]], marker='o', capsize=2)
        c = conf[name]
        a.axhspan(*c['ci'], color='C1', alpha=0.25); a.axhline(c['mean'], color='C1'); a.axhline(0, color='k', lw=0.6)
        a.set_xscale('log', base=2); a.set_title(f"{name}: {t['metric']} ({t['a']} - {t['b']}); {c['verdict']}", fontsize=8)
    figs['confirmatory'] = f
    for name, metrics in (('recovery', ('rmrt', 'reached')), ('beliefs', ('diversity', 'beliefError')),
                          ('secondary', ('discoveries', 'freeRiding', 'findTime', 'overlap', 'followTime', 'commPerWelfare'))):
        f, ax = plt.subplots(1, len(metrics), figsize=(3.6 * len(metrics), 3.2))
        for a, m in zip(np.atleast_1d(ax), metrics):
            _lines(a, main, m, budget='compute')
        np.atleast_1d(ax)[0].legend(fontsize=6); figs[name] = f
    return figs


def sweepFigures(table, opt, sb):
    "Figures for the side sweeps (one panel per factor), study A and study B."
    figs = {}
    if table:
        f, ax = plt.subplots(1, len(table), figsize=(3.4 * len(table), 3.2))
        for a, (factor, lv) in zip(np.atleast_1d(ax), table.items()):
            levels, cols = list(lv), sorted({c for v in lv.values() for c in v if c[1] in (16, 64)})
            for c in cols:
                a.plot(range(len(levels)), [lv[l].get(c, np.nan) for l in levels], marker='o', label=f'{c[0]}/{c[1]}')
            a.set_xticks(range(len(levels)), levels, fontsize=7); a.set_title(factor, fontsize=9)
        np.atleast_1d(ax)[0].set_ylabel('welfare / agent-s'); np.atleast_1d(ax)[0].legend(fontsize=6)
        figs['sweeps'] = f
    if sb:
        Ns = sorted({k.split('/N=')[1] for k in sb})
        f, ax = plt.subplots(1, len(Ns), figsize=(6 * len(Ns), 3.6), squeeze=False)
        for a, N in zip(ax[0], Ns):
            for k, v in sb.items():
                if k.endswith(f'/N={N}'):
                    a.plot(v['curve'], marker='.', ls='--' if k.startswith('fixed') else '-', label=k.split('/N=')[0])
            a.set_xlabel('episode'); a.set_title(f'study B, N = {N}', fontsize=9)
        ax[0][0].set_ylabel('welfare / agent-s'); ax[0][0].legend(fontsize=6)
        figs['studyB'] = f
    return figs

## Tests

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from fastcore.test import test_eq
from explRL.bench_figures import mainFigures, sweepFigures
from explRL.bench_grid import PREREG, mainGrid, sideSweeps, runGrid, confirm
from explRL.bench_report import sweepTable, teamOptimal, studyB

rows = runGrid(mainGrid('quick'), workers=2) + runGrid(sideSweeps('quick'), workers=2)
figs = mainFigures(rows, confirm(rows, seeds=(1000, 1001)), PREREG)
test_eq(set(figs), {'scaling', 'confirmatory', 'recovery', 'beliefs', 'secondary'})
tab = sweepTable(rows)
test_eq(set(sweepFigures(tab, teamOptimal(tab), studyB(rows))), {'sweeps', 'studyB'})
plt.close('all')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()